# P3 DeepSup — RepDWNet + WDice on Kaggle

One notebook, six modes in the first config cell: Control and ex1..ex5. Base f/wdice-loss@d77d518; publish the prepared source to f/DeepSup before Kaggle clones it.

Flow: attach PhenoBench → GPU T4×2 → Save & Run All → one ZIP. P2 recipe: 100 epochs/validation every 5, batch 4/GPU, FP32; main-only validation/inference.

D01: save dataset_audit.json and stop before training if raw label 255 exists. D02/D03: shared rank-zero writer, synchronized validation values, checkpoint-owned history, sanity exclusion and epoch deduplication; acceptance tests run before the campaign.

Validation sampling remains the P2 protocol (drop_last=True); on 772 validation images with 2×4 batches it evaluates 768 images. Full-dataset reporting requires a separate uniform evaluation of all compared checkpoints. Benchmarks use disposable models in separate processes.


In [ ]:
# FIRST CONFIG CELL
REPO_URL = 'https://github.com/FUAVX/RPD_Reproduce.git'
GIT_REF = 'f/DeepSup'
SOURCE_COMMIT = ''  # optional commit SHA reachable from this branch
EXPERIMENT_MODE = 'ex1'  # control / ex1=x3 / ex2=x4 / ex3=x5 / ex4=x6 / ex5=x7
RUN_TAG = 'run1'
DATASET_DIR = ''  # empty: detect attached PhenoBench
NUM_WORKERS = 2
DEVICES = 2
WORKING_ROOT = '/kaggle/working'
RUN_BENCHMARKS = True  # project unified20/100 training,30/200 inference; separate disposable processes
EXPORT_MAIN = True  # export both selected main checkpoints; optional fusion is not requested
EXPERIMENTS = {
    'control': ('P3_DeepSup_Control', None),
    'ex1': ('P3_DeepSup_Ex1_x3', 'x3'),
    'ex2': ('P3_DeepSup_Ex2_x4', 'x4'),
    'ex3': ('P3_DeepSup_Ex3_x5', 'x5'),
    'ex4': ('P3_DeepSup_Ex4_x6', 'x6'),
    'ex5': ('P3_DeepSup_Ex5_x7', 'x7'),
}
if EXPERIMENT_MODE not in EXPERIMENTS:
    raise ValueError('Select control or ex1..ex5')
if DEVICES != 2:
    raise ValueError('P2 runtime protocol requires T4 x2')
SCENARIO, EXPECTED_TAP = EXPERIMENTS[EXPERIMENT_MODE]


In [ ]:
# Kaggle supplies torch/CUDA; retain that installation.
!nvidia-smi
!pip install -q 'numpy<2' 'pytorch-lightning==1.9.5' 'torchmetrics==0.11.4' 'setuptools<70' tensorboard matplotlib pandas tabulate timm scikit-image tifffile albumentations pyyaml rich


In [ ]:
import json
import pathlib
import shutil
import subprocess
import yaml
WORKING_DIR = pathlib.Path(WORKING_ROOT)
REPO_DIR = WORKING_DIR / 'RPD_P2'
RUN_DIR = WORKING_DIR / f'{SCENARIO}_{RUN_TAG}'
if RUN_DIR.exists():
    raise FileExistsError(f'Output already exists: {RUN_DIR}; choose a new RUN_TAG')
def run(command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
run(['git', 'clone', '--branch', GIT_REF, '--single-branch', REPO_URL, str(REPO_DIR)])
if SOURCE_COMMIT:
    run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO_DIR)
BASE_COMMIT = 'd77d518b8f650443216070fa76742db76e41b8ac'
run(['git', 'merge-base', '--is-ancestor', BASE_COMMIT, 'HEAD'], cwd=REPO_DIR)
if (REPO_DIR / 'models/rpdnet/downsampling.py').exists():
    raise RuntimeError('Unexpected Down dispatcher in this campaign; verify f/DeepSup source ancestry')
scenario_path = REPO_DIR / 'configs/p3_deepsup' / f'{SCENARIO}.yaml'
required_paths = [scenario_path] + [REPO_DIR / name for name in (
    'test_wdice.py', 'test_deepsup.py', 'test_validation_infrastructure.py',
    'utils/validation_artifacts.py', 'scripts/check_deepsup_run.py',
    'models/rpdnet/deep_supervision.py',
    'scripts/benchmark_deepsup.py', 'scripts/export_deepsup.py', 'scripts/check_deepsup_data.py',
    'callbacks/deepsup_cost.py', 'utils/tensorboard_compat.py', 'tests/reference/deepsup/pspnet.py')]
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError('DeepSup source files are missing: ' + ', '.join(missing))
def is_phenobench_root(path):
    return (
        (path / 'train' / 'images').is_dir()
        and (path / 'train' / 'semantics').is_dir()
        and (path / 'val' / 'images').is_dir()
        and (path / 'val' / 'semantics').is_dir()
    )

if DATASET_DIR:
    dataset_dir = pathlib.Path(DATASET_DIR)
else:
    kaggle_input = pathlib.Path('/kaggle/input')
    candidates = [
        kaggle_input / 'datasets' / 'ndhoang2310' / 'phenobench-dataset' / 'PhenoBench',
        kaggle_input / 'phenobench-dataset' / 'PhenoBench',
        kaggle_input / 'datasets' / 'ndhoang2310' / 'phenobench-dataset',
        kaggle_input / 'phenobench-dataset',
    ]
    if kaggle_input.is_dir():
        attached_inputs = [p for p in kaggle_input.iterdir() if p.is_dir()]
        candidates.extend(p / 'PhenoBench' for p in attached_inputs)
        candidates.extend(attached_inputs)
    dataset_dir = next((p for p in candidates if is_phenobench_root(p)), None)
if dataset_dir is None or not is_phenobench_root(dataset_dir):
    raise FileNotFoundError('Cannot find a complete PhenoBench root in /kaggle/input. Attach the PhenoBench dataset, or set DATASET_DIR to its root containing train/images, train/semantics, val/images, and val/semantics.')

print(f'[OK] source ref: {subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()}')
print(f'[OK] dataset: {dataset_dir}')
print(f'[OK] train images: {len(list((dataset_dir / "train" / "images").glob("*")))}')


In [ ]:
# Runtime overrides follow the successful WDice notebook.
import copy
import hashlib
RUN_DIR.mkdir(parents=True, exist_ok=True)
cfg = yaml.safe_load(scenario_path.read_text())
base_path = REPO_DIR / 'configs/p2_scenarios/P2_Ex7_RepDWNet_WDice-Loss.yaml'
base_cfg = yaml.safe_load(base_path.read_text())
inherited = copy.deepcopy(cfg)
settings = inherited['backbone'].pop('deep_supervision', None)
inherited['experiment'] = base_cfg['experiment']
assert inherited == base_cfg, 'Recipe drift beyond experiment metadata/DeepSup'
if EXPECTED_TAP is None:
    assert settings is None
else:
    assert settings == {'tap': EXPECTED_TAP, 'weight': 0.4, 'hidden_ratio': 0.5, 'dropout': 0.1, 'align_corners': True}
assert cfg['train']['max_epoch'] == 100
assert cfg['val']['check_val_every_n_epoch'] == 5
assert cfg['train']['precision'] == 32 and cfg['train']['batch_size'] == 4
assert cfg['train']['loss'] == 'weighted_dice'
cfg['data']['path_to_dataset'] = str(dataset_dir)
cfg['data']['num_workers'] = NUM_WORKERS
active_cfg_path = RUN_DIR / f'active_config_{SCENARIO}.yaml'
active_cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
runtime_cfg = copy.deepcopy(cfg)
runtime_cfg['train']['devices'] = DEVICES
runtime_cfg['val']['devices'] = 1  # exact legacy --devices override
runtime_cfg_path = RUN_DIR / 'runtime_config.yaml'
runtime_cfg_path.write_text(yaml.safe_dump(runtime_cfg, sort_keys=False))
print(yaml.safe_dump(runtime_cfg, sort_keys=False))


In [ ]:
# GPU and data guards run before any campaign training.
import torch
if not torch.cuda.is_available() or torch.cuda.device_count() != DEVICES:
    raise RuntimeError('Choose Kaggle GPU T4 x2')
gpu_names = [torch.cuda.get_device_name(i) for i in range(DEVICES)]
if any('T4' not in name for name in gpu_names):
    raise RuntimeError(f'Expected T4 x2, got {gpu_names}')
run(['python', 'scripts/check_deepsup_data.py', '--dataset-dir', str(dataset_dir),
     '--output', str(RUN_DIR / 'dataset_audit.json')], cwd=REPO_DIR)
for script in ['test_wdice.py', 'test_validation_infrastructure.py', 'test_deepsup.py']:
    run(['python', script], cwd=REPO_DIR)
infrastructure_smoke = ['python', 'test_validation_infrastructure.py', '--ddp',
    '--output-dir', str(RUN_DIR / 'smoke/validation_infrastructure'),
    '--accelerator', 'gpu', '--devices', str(DEVICES)]
run(infrastructure_smoke, cwd=REPO_DIR)
run(infrastructure_smoke + ['--resume'], cwd=REPO_DIR)
for mode, (scenario, tap) in EXPERIMENTS.items():
    smoke = ['python', 'test_deepsup.py', '--smoke', '--config',
             str(REPO_DIR / 'configs/p3_deepsup' / f'{scenario}.yaml'),
             '--output-dir', str(RUN_DIR / 'smoke' / mode), '--accelerator', 'gpu',
             '--devices', str(DEVICES)]
    run(smoke, cwd=REPO_DIR)
    run(smoke + ['--resume'], cwd=REPO_DIR)
run(['python', 'test_deepsup.py', '--pipeline-smoke', '--config', str(scenario_path),
     '--output-dir', str(RUN_DIR / 'smoke/pipeline'), '--accelerator', 'gpu',
     '--devices', str(DEVICES)], cwd=REPO_DIR)
# Keep full CLI smoke evidence nested so generic artifact readers see only
# the real campaign training_output/val_history.csv and selected checkpoints.
shutil.make_archive(str(RUN_DIR / 'smoke/pipeline_evidence'), 'zip', root_dir=RUN_DIR / 'smoke/pipeline')
shutil.rmtree(RUN_DIR / 'smoke/pipeline')
run(['python', 'test_deepsup.py', '--native-smoke'], cwd=REPO_DIR)
if RUN_BENCHMARKS:
    # Disposable processes: no weights/BN/optimizer/RNG carry into the campaign.
    run(['python', '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2',
         'scripts/benchmark_deepsup.py', '--mode', 'training', '--device', 'cuda',
         '--output-dir', str(RUN_DIR / 'benchmark')], cwd=REPO_DIR)
    run(['python', 'scripts/benchmark_deepsup.py', '--mode', 'inference', '--device', 'cuda',
         '--output-dir', str(RUN_DIR / 'benchmark')], cwd=REPO_DIR)
TRAIN_LOG_DIR = RUN_DIR / 'training_output'
train_command = ['python', 'multi_metric_train.py', '--config', str(active_cfg_path),
                 '--export_dir', str(TRAIN_LOG_DIR), '--devices', str(DEVICES)]
run(train_command, cwd=REPO_DIR)


In [ ]:
# Require checkpoint/history/monitor parity before exports or completed manifest.
run(['python', 'scripts/check_deepsup_run.py', '--output-dir', str(TRAIN_LOG_DIR),
     '--report', str(RUN_DIR / 'campaign_audit.json')], cwd=REPO_DIR)
# Export main-only checkpoints separately. Fusion is an explicit later audit.
if EXPORT_MAIN:
    for policy in ['best_mIoU', 'best_weedIoU']:
        matches = sorted((TRAIN_LOG_DIR / 'checkpoints').glob(f'*{policy}*.ckpt'))
        if len(matches) != 1:
            raise RuntimeError(f'Expected exactly one {policy} checkpoint: {matches}')
        run(['python', 'scripts/export_deepsup.py', '--config', str(runtime_cfg_path),
             '--checkpoint', str(matches[0]), '--output-dir', str(RUN_DIR / 'main_export' / policy),
             '--device', 'cuda'], cwd=REPO_DIR)
import importlib.metadata
source_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
manifest = {
    'experiment': SCENARIO, 'mode': EXPERIMENT_MODE, 'tap': EXPECTED_TAP, 'run_tag': RUN_TAG,
    'source_repository': REPO_URL, 'requested_ref': GIT_REF, 'requested_commit': SOURCE_COMMIT,
    'source_commit': source_commit, 'base_commit': BASE_COMMIT,
    'base_config': str(base_path.relative_to(REPO_DIR)),
    'base_config_sha256': hashlib.sha256(base_path.read_bytes()).hexdigest(),
    'resolved_config': runtime_cfg, 'train_command': train_command,
    'batch_size_per_GPU': cfg['train']['batch_size'], 'global_batch_size': cfg['train']['batch_size'] * DEVICES,
    'gpu_names': gpu_names, 'bn': 'standard BatchNorm2d, sync_batchnorm=False',
    'seed': cfg['seed'], 'same_production_criterion': True, 'main_only_validation': True,
    'auxiliary': settings, 'head_initialization': 'PyTorch layer defaults, after host in CPU RNG fork/restore; no new seed',
    'official_reference': {'repo': 'https://github.com/hszhao/semseg',
        'commit': '4f274c3f276778228bc14a4565822d46359f0cc8', 'file': 'model/pspnet.py'},
    'benchmarks_requested': RUN_BENCHMARKS, 'main_export_requested': EXPORT_MAIN,
    'versions': {name: importlib.metadata.version(name) for name in
                 ('torch', 'torchvision', 'pytorch-lightning', 'torchmetrics', 'numpy')},
    'validation_infrastructure': {'D01': 'preflight stops on 255; metric semantics unchanged',
        'D02': 'global synchronized metrics; rank-zero persistent writer',
        'D03': 'checkpoint-owned validation history; no sanity rows; epoch deduplication'},
    'validation_sampling': {'drop_last': True, 'batch_size_per_rank': cfg['val']['batch_size'],
        'world_size': DEVICES, 'note': 'P2 sampling retained; independent full validation evaluation is separate'},
    'dataset_audit': json.loads((RUN_DIR / 'dataset_audit.json').read_text()),
    'campaign_audit': json.loads((RUN_DIR / 'campaign_audit.json').read_text()),
    'status': 'completed',
}
required_outputs = [TRAIN_LOG_DIR / 'checkpoints/last.ckpt', TRAIN_LOG_DIR / 'val_history.csv',
                    TRAIN_LOG_DIR / 'training_cost.json', runtime_cfg_path, RUN_DIR / 'dataset_audit.json',
                    RUN_DIR / 'campaign_audit.json']
if EXPECTED_TAP:
    required_outputs.append(TRAIN_LOG_DIR / 'deep_supervision_history.csv')
if EXPORT_MAIN:
    required_outputs.extend(RUN_DIR / 'main_export' / policy / 'main_state_dict.pt'
                            for policy in ['best_mIoU', 'best_weedIoU'])
missing = [str(path) for path in required_outputs if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing completed artifacts: ' + ', '.join(missing))
(RUN_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
archive = shutil.make_archive(str(WORKING_DIR / f'{SCENARIO}_{RUN_TAG}_artifacts'), 'zip', root_dir=RUN_DIR)
print(f'[DONE] Download: {archive}')
print(f'[DONE] Source: {source_commit}')
